# LightGBM — Previsão do IPCA com variáveis externas defasadas

**Projeto:** IPCA sob pressão externa  
**Autor:** Gabriel Abreu  
**Orientadora:** Bruna Nogueira Rezende  

Este notebook cobre as etapas do pipeline analítico para o modelo LightGBM:
1. Engenharia de features (lags, médias móveis, variáveis cíclicas)
2. Prevenção de data leakage
3. Otimização de hiperparâmetros (grid search + blocked time series CV)
4. Expanding window — avaliação fora da amostra (h = 1, 2, 3)
5. Importância de variáveis (gain)
6. Métricas MAE, RMSE, MASE e análise por regime de volatilidade
7. Teste de Diebold-Mariano (ganho incremental do CPI — H3)

### Por que LightGBM para previsão macroeconômica?

O LightGBM (Ke et al., 2017) é um algoritmo de gradient boosting baseado em árvores de decisão com particionamento por folha (*leaf-wise*), o que lhe confere capacidade de capturar relações não lineares e interações entre variáveis sem especificação explícita. Diferentemente do ARIMAX, não pressupõe linearidade nem homocedasticidade. Sua aplicação a séries macroeconômicas requer representação explícita da dimensão temporal via engenharia de features.

## 0. Dependências

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
import pickle

np.random.seed(42)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(f'LightGBM version: {lgb.__version__}')

## 1. Carregamento e preparação base

In [ ]:
df_raw = pd.read_csv('final_df_with_lags.csv', parse_dates=['data'], index_col='data')
df_raw = df_raw.asfreq('MS')

print(f'Shape: {df_raw.shape}')
print(f'Período: {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
print('Colunas:', df_raw.columns.tolist())

## 2. Engenharia de features

O LightGBM não possui estrutura temporal nativa. A dimensão temporal é representada por:

| Feature               | Descrição                                              | Justificativa                              |
|-----------------------|-------------------------------------------------------|--------------------------------------------|
| `ipca_lag_1..6`       | Lags 1–6 do IPCA                                      | Componente autorregressivo                  |
| `usd_brl_lag4`        | Câmbio com defasagem ótima (CCF)                      | Pass-through cambial                        |
| `us_cpi_lag2/3`       | CPI EUA com defasagem ótima                           | Inflação importada                          |
| `roll_mean_3/6_*`     | Média móvel deslocada de 3 e 6 meses                 | Tendência de curto prazo                   |
| `month_sin/cos`       | Sazonalidade cíclica: sen(2π·m/12), cos(2π·m/12)    | Captura continuidade dez→jan               |

**Prevenção de data leakage:** todas as features são construídas com deslocamento explícito. A linha `t` contém apenas valores disponíveis até `t − 1` (ou `t − lag` para lags maiores). Médias móveis usam `shift(1)` antes do `rolling`, garantindo que a média de 3 meses em `t` use `t−1, t−2, t−3`.

In [ ]:
def build_features(df, target='ipca_pct', cpi_lag=2, fx_lag=4):
    """
    Constrói matriz de features para o LightGBM.
    Todas as features são deslocadas para evitar data leakage.
    
    Parâmetros
    ----------
    df       : DataFrame com séries em variação percentual
    target   : coluna alvo (IPCA)
    cpi_lag  : defasagem ótima do CPI (identificada via CCF/AIC)
    fx_lag   : defasagem ótima do câmbio
    
    Retorna
    -------
    feat_df  : DataFrame de features sem vazamento de informação futura
    """
    feat = pd.DataFrame(index=df.index)

    # Lags do IPCA (1 a 6 meses)
    for lag in range(1, 7):
        feat[f'ipca_lag_{lag}'] = df[target].shift(lag)

    # Câmbio e CPI com defasagens ótimas
    feat[f'usd_brl_lag{fx_lag}']  = df['usd_brl_pct'].shift(fx_lag)
    feat[f'us_cpi_lag{cpi_lag}']  = df['us_cpi_pct'].shift(cpi_lag)
    feat['us_cpi_lag3']            = df['us_cpi_pct'].shift(3)

    # Médias móveis deslocadas (shifted rolling mean — sem leakage)
    for window in [3, 6]:
        feat[f'ipca_roll_mean_{window}']    = df[target].shift(1).rolling(window).mean()
        feat[f'usd_brl_roll_mean_{window}'] = df['usd_brl_pct'].shift(fx_lag).rolling(window).mean()
        feat[f'us_cpi_roll_mean_{window}']  = df['us_cpi_pct'].shift(cpi_lag).rolling(window).mean()

    # Sazonalidade cíclica (continuidade dez → jan)
    month = df.index.month
    feat['month_sin'] = np.sin(2 * np.pi * month / 12)
    feat['month_cos'] = np.cos(2 * np.pi * month / 12)

    return feat


feat_full = build_features(df_raw)
print(f'Features construídas: {feat_full.shape[1]} variáveis')
print(feat_full.columns.tolist())
print(f'\nPrimeiras linhas com NaN (lags): {feat_full.isna().any(axis=1).sum()} linhas')

In [ ]:
# ── Esquema temporal das features (prevenção de leakage) ──
print("""
Esquema de disponibilidade temporal das features:

  Período t:   [resultado a prever: IPCA_t]
               ↑
  Disponível em t-1: ipca_lag_1, usd_brl_lag1, us_cpi_lag1
  Disponível em t-4: usd_brl_lag4  (lag ótimo câmbio)
  Disponível em t-2: us_cpi_lag2   (lag ótimo CPI)

  Para h=2: feature mais recente do IPCA é ipca_lag_2 (não ipca_lag_1)
  Para h=3: feature mais recente do IPCA é ipca_lag_3
  → Modelos separados por horizonte garantem essa restrição.
""")

## 3. Features por horizonte de previsão

Para h = 2 e h = 3, os lags com defasagem menor que h são removidos (não disponíveis em t ao prever t+h).

In [ ]:
def get_feature_cols(feat_df, horizon, cpi_lag=2, fx_lag=4):
    """
    Retorna colunas de features válidas para o horizonte h.
    Remove lags < h (indisponíveis no momento da previsão).
    """
    cols = []
    for col in feat_df.columns:
        # Lags do IPCA: apenas lags >= horizon
        if col.startswith('ipca_lag_'):
            lag = int(col.split('_')[-1])
            if lag >= horizon:
                cols.append(col)
        # Câmbio: lag fixo >= horizon
        elif col.startswith('usd_brl_lag'):
            lag = int(col.replace('usd_brl_lag', ''))
            if lag >= horizon:
                cols.append(col)
        # CPI: lag fixo >= horizon
        elif col.startswith('us_cpi_lag'):
            lag = int(col.replace('us_cpi_lag', ''))
            if lag >= horizon:
                cols.append(col)
        # Médias móveis e sazonalidade: sempre incluídas (calculadas com shift adequado)
        else:
            cols.append(col)
    return cols


for h in [1, 2, 3]:
    cols = get_feature_cols(feat_full, h)
    print(f'h={h}: {len(cols)} features → {cols}')

## 4. Divisão treino/teste e blocked time series CV

In [ ]:
TARGET = 'ipca_pct'

# Combinar target + features, remover NaN iniciais
data = pd.concat([df_raw[[TARGET]], feat_full], axis=1).dropna()

train_data = data[data.index <  '2020-01-01']
test_data  = data[data.index >= '2020-01-01']

print(f'Treino: {train_data.index[0].date()} → {train_data.index[-1].date()} ({len(train_data)} obs)')
print(f'Teste:  {test_data.index[0].date()}  → {test_data.index[-1].date()}  ({len(test_data)} obs)')

In [ ]:
def blocked_ts_split(n, n_splits=5):
    """
    Blocked time series split: divide em n_splits folds consecutivos,
    preservando a ordem cronológica (sem sobreposição entre folds).
    Retorna lista de (train_idx, val_idx).
    """
    fold_size = n // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = i * fold_size
        val_end   = train_end + fold_size
        splits.append((np.arange(0, train_end), np.arange(train_end, val_end)))
    return splits


# Visualização do esquema de validação
fig, ax = plt.subplots(figsize=(10, 3))
splits = blocked_ts_split(len(train_data), n_splits=5)
colors = ['steelblue', 'darkorange']

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    ax.barh(fold_i, len(tr_idx),  left=0,          color=colors[0], alpha=0.7, height=0.6)
    ax.barh(fold_i, len(val_idx), left=len(tr_idx), color=colors[1], alpha=0.7, height=0.6)

ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Observações (meses)')
ax.set_title('Blocked Time Series CV — 5 folds')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=colors[0], label='Treino'),
    Patch(color=colors[1], label='Validação'),
])
plt.tight_layout()
plt.savefig('lgbm_cv_schema.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Otimização de hiperparâmetros (grid search)

Busca sobre o espaço definido na metodologia. Métrica de otimização: MAE (consistência com métrica principal de avaliação).

In [ ]:
PARAM_GRID = {
    'learning_rate':  [0.01, 0.05, 0.1],
    'n_estimators':   [100, 200, 300, 500],
    'max_depth':      [3, 5, 7],
    'num_leaves':     [15, 31, 63],  # num_leaves = 2^max_depth - 1 como referência
}

def grid_search_lgbm(X_train, y_train, param_grid, n_splits=5):
    """
    Grid search com blocked time series CV para LightGBM.
    Retorna os melhores hiperparâmetros e DataFrame com todos os resultados.
    """
    splits   = blocked_ts_split(len(X_train), n_splits)
    best_mae = np.inf
    best_params = None
    all_results = []

    for params in ParameterGrid(param_grid):
        fold_maes = []
        for tr_idx, val_idx in splits:
            X_tr, y_tr   = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

            model = lgb.LGBMRegressor(
                **params,
                random_state=42,
                verbose=-1,
            )
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=50, verbose=False),
                    lgb.log_evaluation(period=-1),
                ],
            )
            fold_maes.append(mean_absolute_error(y_val, model.predict(X_val)))

        cv_mae = np.mean(fold_maes)
        all_results.append({**params, 'cv_mae': cv_mae})

        if cv_mae < best_mae:
            best_mae    = cv_mae
            best_params = params

    return best_params, best_mae, pd.DataFrame(all_results).sort_values('cv_mae')


# Otimização para h=1 (hiperparâmetros fixados e reutilizados em todos os horizontes)
feat_cols_h1 = get_feature_cols(feat_full, horizon=1)
X_tr_h1 = train_data[feat_cols_h1]
y_tr_h1 = train_data[TARGET]

print('Rodando grid search (pode demorar)...')
best_params, best_cv_mae, grid_results = grid_search_lgbm(X_tr_h1, y_tr_h1, PARAM_GRID)
print(f'\nMelhores hiperparâmetros: {best_params}')
print(f'CV MAE: {best_cv_mae:.4f}')
print('\nTop 5:')
print(grid_results.head(5).to_string(index=False))

## 6. Expanding window — avaliação fora da amostra (h = 1, 2, 3)

Hiperparâmetros fixados na otimização acima. Modelo reestimado a cada passo da expanding window.

In [ ]:
def mase(actual, predicted, train_actual):
    naive_mae = np.mean(np.abs(np.diff(train_actual)))
    return np.mean(np.abs(actual - predicted)) / naive_mae


def expanding_window_lgbm(data, feat_cols, target, horizon, params,
                           train_end='2019-12-01'):
    """
    Expanding window para LightGBM em horizonte h.
    Para cada t no período de teste, treina com todos os dados até t-h
    e prevê o valor em t.
    """
    test_idx = data[data.index > train_end].index
    preds, actuals = [], []

    for t in test_idx:
        cutoff = t - pd.DateOffset(months=horizon)
        train_w = data[data.index <= cutoff]

        if len(train_w) < 36:
            continue

        X_tr = train_w[feat_cols]
        y_tr = train_w[target]
        X_te = data.loc[[t], feat_cols]

        model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
        model.fit(X_tr, y_tr)

        preds.append(model.predict(X_te)[0])
        actuals.append(data.loc[t, target])

    return np.array(actuals), np.array(preds)


# Duas especificações: sem CPI (M1) e com CPI (M2) — para teste H3
def get_fx_only_cols(feat_cols):
    """Remove colunas de CPI para a especificação M1 (só câmbio)."""
    return [c for c in feat_cols if 'us_cpi' not in c]


results_ew = {}
print('Iniciando expanding window LightGBM...')

for h in [1, 2, 3]:
    feat_cols = get_feature_cols(feat_full, horizon=h)
    feat_cols_fx = get_fx_only_cols(feat_cols)

    # M1: só câmbio
    act, pred = expanding_window_lgbm(data, feat_cols_fx, TARGET, h, best_params)
    results_ew[('M1_fx', h)] = {'actuals': act, 'preds': pred}
    print(f'  h={h} | M1_fx ✓')

    # M2: câmbio + CPI
    act, pred = expanding_window_lgbm(data, feat_cols, TARGET, h, best_params)
    results_ew[('M2_fx_cpi', h)] = {'actuals': act, 'preds': pred}
    print(f'  h={h} | M2_fx_cpi ✓')

print('Expanding window concluída.')

## 7. Métricas de avaliação

In [ ]:
train_ipca = data[data.index < '2020-01-01'][TARGET].values

rows = []
for (model_name, h), d in results_ew.items():
    act, pred = d['actuals'], d['preds']
    mask = ~np.isnan(pred)
    rows.append({
        'model':   model_name,
        'horizon': h,
        'MAE':     mean_absolute_error(act[mask], pred[mask]),
        'RMSE':    np.sqrt(mean_squared_error(act[mask], pred[mask])),
        'MASE':    mase(act[mask], pred[mask], train_ipca),
        'n':       mask.sum(),
    })

metrics_df = pd.DataFrame(rows).sort_values(['horizon', 'MAE'])
print('Métricas LightGBM — expanding window:')
print(metrics_df.round(4).to_string(index=False))

## 8. Importância de variáveis (gain)

A importância por `gain` mensura o ganho total de informação que cada feature proporciona ao longo de todas as árvores. Permite identificar quais lags e variáveis externas mais contribuem para a previsão.

In [ ]:
# Treinar modelo final sobre toda a janela de treino para análise de importância
feat_cols_h1 = get_feature_cols(feat_full, horizon=1)

model_final = lgb.LGBMRegressor(**best_params, random_state=42, verbose=-1)
model_final.fit(train_data[feat_cols_h1], train_data[TARGET])

importance_df = pd.DataFrame({
    'feature':    feat_cols_h1,
    'importance': model_final.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
colors = [
    'seagreen'    if 'cpi' in f    else
    'darkorange'  if 'brl' in f    else
    'steelblue'
    for f in importance_df['feature']
]
ax.barh(importance_df['feature'], importance_df['importance'], color=colors)
ax.set_xlabel('Importância (gain)')
ax.set_title('LightGBM — Importância de variáveis (h=1)')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue',   label='Lags IPCA'),
    Patch(color='darkorange',  label='Câmbio USD/BRL'),
    Patch(color='seagreen',    label='CPI EUA'),
])
plt.tight_layout()
plt.savefig('lgbm_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nImportância das variáveis de CPI (gain):')
print(importance_df[importance_df['feature'].str.contains('cpi')].to_string(index=False))

## 9. Análise por regime de volatilidade cambial (H2)

In [ ]:
df_raw['vol_cambio_12m'] = df_raw['usd_brl_pct'].rolling(12).std()
vol_threshold = df_raw[df_raw.index < '2020-01-01']['vol_cambio_12m'].median()
df_raw['regime_vol'] = np.where(df_raw['vol_cambio_12m'] > vol_threshold, 'alta', 'baixa')

print(f'Limiar de volatilidade (mediana treino): {vol_threshold:.4f}')

test_index = data[data.index >= '2020-01-01'].index

rows_regime = []
for (model_name, h), d in results_ew.items():
    act, pred = d['actuals'], d['preds']
    n = min(len(act), len(test_index))
    regimes = df_raw.loc[test_index[:n], 'regime_vol'].values

    for regime in ['alta', 'baixa']:
        mask = (regimes == regime) & ~np.isnan(pred[:n])
        if mask.sum() < 3:
            continue
        rows_regime.append({
            'model':   model_name,
            'horizon': h,
            'regime':  regime,
            'MAE':     mean_absolute_error(act[:n][mask], pred[:n][mask]),
            'RMSE':    np.sqrt(mean_squared_error(act[:n][mask], pred[:n][mask])),
            'n':       mask.sum(),
        })

regime_df = pd.DataFrame(rows_regime).sort_values(['horizon', 'regime', 'MAE'])
print('\nMétricas por regime de volatilidade:')
print(regime_df.round(4).to_string(index=False))

## 10. Teste de Diebold-Mariano (H3 — ganho incremental do CPI)

In [ ]:
def diebold_mariano(e1, e2, h=1):
    """Teste DM (1995). e1, e2: erros de previsão. H0: acurácia equivalente."""
    d       = e1**2 - e2**2
    n       = len(d)
    d_mean  = np.mean(d)
    gamma0  = np.var(d, ddof=1)
    gammas  = [np.cov(d[k:], d[:-k])[0, 1] for k in range(1, h)]
    var_d   = max((gamma0 + 2 * sum(gammas)) / n, 1e-10)
    dm_stat = d_mean / np.sqrt(var_d)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {'dm_stat': dm_stat, 'p_value': p_value}


print('Teste Diebold-Mariano — M1 (só câmbio) vs M2 (câmbio + CPI)')
print('DM < 0 → M2 melhor | p < 0.05 → rejeita H0')
print('-' * 65)

for h in [1, 2, 3]:
    m1_d = results_ew.get(('M1_fx', h))
    m2_d = results_ew.get(('M2_fx_cpi', h))
    if not m1_d or not m2_d:
        continue

    act   = m1_d['actuals']
    e1    = act - m1_d['preds']
    e2    = act - m2_d['preds']
    mask  = ~(np.isnan(e1) | np.isnan(e2))
    dm    = diebold_mariano(e1[mask], e2[mask], h=h)

    sig = '✓ SIGNIFICATIVO (5%)' if dm['p_value'] < 0.05 else '✗ não significativo'
    print(f'  h={h}: DM={dm["dm_stat"]:+.4f}, p={dm["p_value"]:.4f}  → {sig}')

## 11. Gráficos: observado vs. previsto

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
test_dates = data[data.index >= '2020-01-01'].index

for i, h in enumerate([1, 2, 3]):
    ax   = axes[i]
    act  = results_ew[('M1_fx', h)]['actuals']
    p_m1 = results_ew[('M1_fx', h)]['preds']
    p_m2 = results_ew[('M2_fx_cpi', h)]['preds']

    n     = min(len(act), len(test_dates))
    dates = test_dates[:n]

    ax.plot(dates, act[:n],  color='black',      linewidth=1.2,  label='Realizado')
    ax.plot(dates, p_m1[:n], color='steelblue',  linewidth=0.9, linestyle='--', label='M1 — câmbio')
    ax.plot(dates, p_m2[:n], color='darkorange', linewidth=0.9, linestyle='--', label='M2 — câmbio+CPI')
    ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-12-01'),
               alpha=0.07, color='red', label='Choque COVID')
    ax.axhline(0, color='gray', linewidth=0.4)
    ax.set_title(f'LightGBM — h={h} mês(es) à frente', fontsize=11)
    ax.set_ylabel('Var. % IPCA')
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.suptitle('LightGBM — Previsões out-of-sample vs. realizadas (2020–2024)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('lgbm_ew_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Exportação

In [ ]:
with open('lgbm_best_params.pkl', 'wb') as f:
    pickle.dump(best_params, f)

with open('lgbm_model_final.pkl', 'wb') as f:
    pickle.dump(model_final, f)

metrics_df.to_csv('lgbm_metrics.csv', index=False)
regime_df.to_csv('lgbm_metrics_regime.csv', index=False)
importance_df.to_csv('lgbm_feature_importance.csv', index=False)

for (model_name, h), d in results_ew.items():
    n = min(len(d['actuals']), len(test_dates))
    pd.DataFrame({
        'date':      test_dates[:n],
        'actual':    d['actuals'][:n],
        'predicted': d['preds'][:n],
    }).to_csv(f'lgbm_preds_{model_name}_h{h}.csv', index=False)

print('Exportação concluída.')